In [25]:
"""Extract email body to Excel + Save .msg files
Sheet 1: RTA Email Body     — RTA team -> CNX.EXPEDIA.IDAdmins
Sheet 2: IDAdmins Reply     — CNX.EXPEDIA.IDAdmins replies
Sheet 3: Operation Alert    — Attrition Notification emails
MSG out : DEACTIVATE_ACCOUNT / RTA_SEND | IDADMINS_REPLY | OPERATION_ALERT / YY_MM / *.msg
"""

from __future__ import annotations

import win32com.client as win32
import re
from pathlib import Path
from datetime import date, datetime
from typing import List, Dict, Any, Optional
from bs4 import BeautifulSoup
import xlsxwriter

# ====================== CONFIG ======================
ACCOUNT_NAME        = "huuchinh.nguyen@concentrix.com"

# [1] RTA
RTA_FOLDER_NAME     = "RTA team"
RTA_SENDERS         = {
    "huuchinh.nguyen@concentrix.com",
    "duonghoangvu.pham@concentrix.com",
    "hoangminhduc.nguyen@concentrix.com",
}
RTA_SENDER_NAMES    = {
    "huu chinh nguyen",
    "duong hoang vu pham",
    "hoang minh duc nguyen",
}
TARGET_RECIPIENT    = "CNX.EXPEDIA.IDAdmins"

# [2] IDAdmins
IDADMINS_FOLDER     = "GC3 + ExpAdmin"
IDADMINS_SENDER     = "CNX.EXPEDIA.IDAdmins"

# [3] Alert
ALERT_SUBJECT_KEYWORDS = [
    "Expedia VN_ Attrition Notification & Delete Okta Account",
    "ATTRITION NOTIFICATION",
]

# Shared
SUBJECT_KEYWORD     = "Account Deactivation Request"
FILTER_FROM_DATE    = date(2026, 1, 1)
FILTER_TO_DATE      = date.today()

# Output
OUTPUT_FILE         = Path("email_body_extract.xlsx")
SAVE_MSG            = True
MSG_BASE            = Path(r"C:\Users\ADMIN\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\DEACTIVATE_ACCOUNT")
MSG_FOLDERS         = {
    "rta"     : MSG_BASE / "RTA_SEND",
    "idadmins": MSG_BASE / "IDADMINS_REPLY",
    "alert"   : MSG_BASE / "OPERATION_ALERT",
}


# ====================== STEP 1: CONNECT ======================
app     = win32.Dispatch("Outlook.Application")
ns      = app.GetNamespace("MAPI")
account = ns.Folders(ACCOUNT_NAME)

print(f"Connected  : {ACCOUNT_NAME}")
print(f"Date range : {FILTER_FROM_DATE} -> {FILTER_TO_DATE}")
print(f"Save MSG   : {SAVE_MSG} -> {MSG_BASE}")
print("-" * 60)

# Create MSG output directories
if SAVE_MSG:
    for folder in MSG_FOLDERS.values():
        folder.mkdir(parents=True, exist_ok=True)


# ====================== STEP 2: HELPERS ======================
def normalize_dt(dt) -> Optional[datetime]:
    return dt.replace(tzinfo=None) if dt else None


def get_month_folder(d: date) -> str:
    return f"{str(d.year)[-2:]}_{d.month:02d}"


def find_folder(root, target_name: str):
    if not hasattr(root, "Folders"):
        return None
    for folder in root.Folders:
        if folder.Name == target_name:
            return folder
        found = find_folder(folder, target_name)
        if found:
            return found
    return None


def resolve_sender_name(item) -> str:
    try:
        sender = item.SenderName or ""
        if not sender or "/O=" in sender.upper():
            ex_user = item.Sender.GetExchangeUser() if item.SenderEmailType == "EX" else None
            sender  = ex_user.PrimarySmtpAddress if ex_user else item.SenderEmailAddress
        return sender.strip()
    except Exception:
        return "[Unknown]"


def resolve_sender_email(item) -> str:
    try:
        addr = item.SenderEmailAddress or ""
        if "@" in addr:
            return addr.lower()
        ex_user = item.Sender.GetExchangeUser() if item.SenderEmailType == "EX" else None
        return (ex_user.PrimarySmtpAddress or "").lower() if ex_user else ""
    except Exception:
        return (item.SenderName or "").lower()


def is_sent_to_idadmins(item) -> bool:
    try:
        return any(
            TARGET_RECIPIENT.lower() in (r.Name or "").lower()
            for r in item.Recipients
        )
    except Exception:
        return False


def has_alert_subject(subject: str) -> bool:
    s = subject.lower()
    return any(kw.lower() in s for kw in ALERT_SUBJECT_KEYWORDS)


def normalize_header(h: str) -> str:
    h = h.strip()
    if re.search(r'emp\s*(id|ID)|employee\s*(id|ID)|oracle\s*id', h, re.IGNORECASE):
        return "Employee ID"
    if re.search(r'termination\s+date|lwd|last\s+work', h, re.IGNORECASE):
        return "Termination Date"
    if re.search(r'^email', h, re.IGNORECASE):
        return "Email"
    if re.search(r'office\s*id|user\s*id|people\s*id|iex\s*id|employee\s*name', h, re.IGNORECASE):
        return "__DROP__"
    return h


def parse_tables(item) -> List[Dict[str, str]]:
    KEEP_COLS = {"Employee ID", "Termination Date", "Email"}
    try:
        soup   = BeautifulSoup(item.HTMLBody or "", "html.parser")
        result = []
        for table in soup.find_all("table"):
            headers = [normalize_header(th.get_text(strip=True)) for th in table.find_all("th")]
            all_tr  = table.find_all("tr")
            if not headers and all_tr:
                headers = [normalize_header(td.get_text(strip=True)) for td in all_tr[0].find_all(["td", "th"])]
                all_tr  = all_tr[1:]
            if not headers or not any(h in KEEP_COLS for h in headers):
                continue
            for tr in all_tr:
                cells = [td.get_text(strip=True) for td in tr.find_all("td")]
                if len(cells) != len(headers) or not any(cells):
                    continue
                row      = dict(zip(headers, cells))
                filtered = {k: v for k, v in row.items() if k in KEEP_COLS}
                if filtered:
                    result.append(filtered)
        return result
    except Exception:
        return []


def parse_plain_text(item) -> str:
    try:
        soup = BeautifulSoup(item.HTMLBody or "", "html.parser")
        for table in soup.find_all("table"):
            table.decompose()
        lines = [ln.strip() for ln in soup.get_text(separator="\n").splitlines()]
        return "\n".join(ln for ln in lines if ln)
    except Exception:
        return (item.Body or "").strip()


def extract_empid_from_subject(subject: str) -> Optional[str]:
    id_match = re.search(r'(?:EID:?\s*)?(1\d{7,9})(?!\d)', subject, flags=re.IGNORECASE)
    return id_match.group(1) if id_match else None


def save_msg(item, received: datetime, subject: str, category: str) -> str:
    """Save email as .msg to MSG_FOLDERS[category]/YY_MM/. Skip if exists."""
    if not SAVE_MSG:
        return ""
    save_dir = MSG_FOLDERS[category] / get_month_folder(received.date())
    save_dir.mkdir(parents=True, exist_ok=True)

    safe     = re.sub(r'[\\/:*?"<>|]', "_", subject)
    safe     = re.sub(r"\s+", " ", safe).strip()[:150] or "No_Subject"
    filename = f"{safe}.msg"
    path     = save_dir / filename

    if path.exists():
        return filename
    try:
        item.SaveAs(str(path), 3)
        return filename
    except Exception as e:
        print(f"   !! MSG save failed: {filename} -> {e}")
        return ""


def make_record(item, subject: str, received: datetime,
                table_row: Optional[Dict] = None,
                saved_file: str = "") -> Dict[str, Any]:
    return {
        "Subject"               : subject,
        "SentDatetime"          : received,
        "SenderName"            : resolve_sender_name(item),
        "SenderEmail"           : resolve_sender_email(item),
        "BodyText"              : parse_plain_text(item),
        "Table_Employee ID"     : table_row.get("Employee ID",      "") if table_row else None,
        "Table_Termination Date": table_row.get("Termination Date",  "") if table_row else None,
        "Table_Email"           : table_row.get("Email",             "") if table_row else None,
        "SavedFile"             : saved_file,
    }


def dedup_records(raw: List[Dict[str, Any]], label: str) -> List[Dict[str, Any]]:
    seen: set = set()
    out:  List[Dict[str, Any]] = []
    for rec in raw:
        emp_id_raw = rec.get("Table_Employee ID", "")
        try:
            emp_id_int = int(str(emp_id_raw).strip()) if emp_id_raw else None
        except (ValueError, TypeError):
            emp_id_int = None
        rec["Table_Employee ID"] = emp_id_int
        if emp_id_int is None:
            out.append(rec)
            continue
        if emp_id_int not in seen:
            seen.add(emp_id_int)
            out.append(rec)
        else:
            print(f"   -- [{label}] Dup: EmpID={emp_id_int} | {rec['SentDatetime'].strftime('%Y-%m-%d')} | {rec['Subject'][:45]}")
    print(f"   [{label}] {len(raw):,} raw -> {len(out):,} after dedup ({len(raw)-len(out):,} removed)")
    return out


# ====================== STEP 3: COLLECT RTA ======================
print(f"\n[Sheet 1] RTA emails in '{RTA_FOLDER_NAME}'...")
rta_folder = find_folder(account, RTA_FOLDER_NAME)
if rta_folder is None:
    raise ValueError(f"Folder '{RTA_FOLDER_NAME}' not found!")
print(f"   Found: {rta_folder.Items.Count:,} items")

restrict_deact = (
    f"@SQL=\"urn:schemas:httpmail:subject\" LIKE '%{SUBJECT_KEYWORD}%' "
    f"AND \"urn:schemas:httpmail:datereceived\" >= '{FILTER_FROM_DATE.strftime('%m/%d/%Y')}' "
    f"AND \"urn:schemas:httpmail:datereceived\" <= '{FILTER_TO_DATE.strftime('%m/%d/%Y')}'"
)

try:
    rta_items = rta_folder.Items.Restrict(restrict_deact)
    print(f"   After filter: {rta_items.Count:,} items")
except Exception as e:
    print(f"   Restrict failed ({e}), fallback")
    rta_items = rta_folder.Items

rta_raw: List[Dict[str, Any]] = []

for item in rta_items:
    if not (hasattr(item, "Subject") and hasattr(item, "ReceivedTime")):
        continue
    received = normalize_dt(item.ReceivedTime)
    if not received or not (FILTER_FROM_DATE <= received.date() <= FILTER_TO_DATE):
        continue

    subject      = item.Subject or ""
    sender_email = resolve_sender_email(item)
    sender_name  = resolve_sender_name(item)

    if subject.upper().startswith(("RE:", "FW:")):
        continue
    if SUBJECT_KEYWORD.lower() not in subject.lower():
        continue

    is_rta = (
        sender_email in {s.lower() for s in RTA_SENDERS} or
        any(s.split("@")[0].lower() in sender_email for s in RTA_SENDERS) or
        sender_name.lower() in RTA_SENDER_NAMES
    )
    if not is_rta or not is_sent_to_idadmins(item):
        continue

    saved      = save_msg(item, received, subject, "rta")
    table_rows = parse_tables(item)

    if table_rows:
        for row in table_rows:
            rta_raw.append(make_record(item, subject, received, row, saved))
    else:
        rta_raw.append(make_record(item, subject, received, None, saved))

    print(f"   ++ {received.strftime('%Y-%m-%d')} | {sender_name} | {subject[:55]}")

rta_raw.sort(key=lambda r: r["SentDatetime"])
rta_deduped = dedup_records(rta_raw, "RTA")


# ====================== STEP 4: COLLECT IDADMINS ======================
print(f"\n[Sheet 2] IDAdmins emails in '{IDADMINS_FOLDER}'...")
idadmins_folder = find_folder(account, IDADMINS_FOLDER)
if idadmins_folder is None:
    raise ValueError(f"Folder '{IDADMINS_FOLDER}' not found!")
print(f"   Found: {idadmins_folder.Items.Count:,} items")

try:
    idadmins_items = idadmins_folder.Items.Restrict(restrict_deact)
    print(f"   After filter: {idadmins_items.Count:,} items")
except Exception as e:
    print(f"   Restrict failed ({e}), fallback")
    idadmins_items = idadmins_folder.Items

idadmins_raw: List[Dict[str, Any]] = []

for item in idadmins_items:
    if not (hasattr(item, "Subject") and hasattr(item, "ReceivedTime")):
        continue
    received = normalize_dt(item.ReceivedTime)
    if not received or not (FILTER_FROM_DATE <= received.date() <= FILTER_TO_DATE):
        continue

    subject     = item.Subject or ""
    sender_name = resolve_sender_name(item)

    if SUBJECT_KEYWORD.lower() not in subject.lower():
        continue
    if IDADMINS_SENDER.lower() not in sender_name.lower():
        continue

    saved      = save_msg(item, received, subject, "idadmins")
    table_rows = parse_tables(item)

    if table_rows:
        for row in table_rows:
            idadmins_raw.append(make_record(item, subject, received, row, saved))
    else:
        idadmins_raw.append(make_record(item, subject, received, None, saved))

    print(f"   ++ {received.strftime('%Y-%m-%d')} | {sender_name} | {subject[:55]}")

idadmins_raw.sort(key=lambda r: r["SentDatetime"])
idadmins_deduped = dedup_records(idadmins_raw, "IDAdmins")


# ====================== STEP 5: COLLECT ALERT ======================
print(f"\n[Sheet 3] Operation Alert emails (full mailbox)...")

alert_raw:    List[Dict[str, Any]] = []
alert_dasl    = "@SQL=\"urn:schemas:httpmail:subject\" LIKE '%Attrition%'"
folders_queue = [account]

while folders_queue:
    cur = folders_queue.pop(0)
    if not hasattr(cur, "Items") or not hasattr(cur, "Name"):
        continue
    try:
        items = cur.Items.Restrict(alert_dasl)
    except Exception:
        items = cur.Items

    for item in items:
        if not (hasattr(item, "Subject") and hasattr(item, "ReceivedTime")):
            continue
        subject = item.Subject or ""
        if subject.upper().startswith(("RE:", "FW:")):
            continue
        if not has_alert_subject(subject):
            continue
        received = normalize_dt(item.ReceivedTime)
        if not received or not (FILTER_FROM_DATE <= received.date() <= FILTER_TO_DATE):
            continue

        sender_name = resolve_sender_name(item)
        saved       = save_msg(item, received, subject, "alert")
        table_rows  = parse_tables(item)

        if table_rows:
            for row in table_rows:
                alert_raw.append(make_record(item, subject, received, row, saved))
        else:
            rec = make_record(item, subject, received, None, saved)
            rec["Table_Employee ID"] = extract_empid_from_subject(subject)
            alert_raw.append(rec)

        print(f"   ++ {received.strftime('%Y-%m-%d')} | {sender_name} | {subject[:55]}")

    if hasattr(cur, "Folders"):
        for sf in cur.Folders:
            folders_queue.append(sf)

alert_raw.sort(key=lambda r: r["SentDatetime"])
alert_deduped = dedup_records(alert_raw, "Alert")


# ====================== STEP 6: EXPORT EXCEL ======================
print(f"\nExporting -> {OUTPUT_FILE}...")

workbook = xlsxwriter.Workbook(str(OUTPUT_FILE))

fmts = {
    "dt"        : workbook.add_format({"num_format": "yyyy-mm-dd hh:mm", "border": 1, "valign": "top"}),
    "dt_alt"    : workbook.add_format({"num_format": "yyyy-mm-dd hh:mm", "border": 1, "valign": "top", "bg_color": "#F2F2F2"}),
    "wrap"      : workbook.add_format({"text_wrap": True, "valign": "top", "border": 1}),
    "wrap_alt"  : workbook.add_format({"text_wrap": True, "valign": "top", "border": 1, "bg_color": "#F2F2F2"}),
    "normal"    : workbook.add_format({"valign": "top", "border": 1}),
    "normal_alt": workbook.add_format({"valign": "top", "border": 1, "bg_color": "#F2F2F2"}),
    "num"       : workbook.add_format({"valign": "top", "border": 1, "num_format": "0"}),
    "num_alt"   : workbook.add_format({"valign": "top", "border": 1, "num_format": "0", "bg_color": "#F2F2F2"}),
}

ALL_COLS = [
    "Subject",
    "SentDatetime",
    "SenderName",
    "SenderEmail",
    "BodyText",
    "Table_Employee ID",
    "Table_Termination Date",
    "Table_Email",
    "SavedFile",
]

COL_WIDTHS = {
    "Subject"               : 55,
    "SentDatetime"          : 20,
    "SenderName"            : 25,
    "SenderEmail"           : 35,
    "BodyText"              : 60,
    "Table_Employee ID"     : 15,
    "Table_Termination Date": 20,
    "Table_Email"           : 35,
    "SavedFile"             : 55,
}


def write_sheet(wb, records: List[Dict[str, Any]], sheet_name: str, header_color: str):
    ws      = wb.add_worksheet(sheet_name)
    hdr_fmt = wb.add_format({
        "bold": True, "bg_color": header_color,
        "font_color": "white", "border": 1, "valign": "vcenter",
    })

    for col_idx, col_name in enumerate(ALL_COLS):
        ws.write(0, col_idx, col_name, hdr_fmt)
        ws.set_column(col_idx, col_idx, COL_WIDTHS.get(col_name, 22))

    for row_idx, rec in enumerate(records, start=1):
        alt = row_idx % 2
        ws.set_row(row_idx, 60)

        for col_idx, col_name in enumerate(ALL_COLS):
            val = rec.get(col_name)

            if isinstance(val, datetime):
                fmt = fmts["dt_alt"]     if alt else fmts["dt"]
            elif col_name == "BodyText":
                fmt = fmts["wrap_alt"]   if alt else fmts["wrap"]
            elif col_name == "Table_Employee ID":
                fmt = fmts["num_alt"]    if alt else fmts["num"]
            else:
                fmt = fmts["normal_alt"] if alt else fmts["normal"]

            if val is None:
                ws.write(row_idx, col_idx, "", fmt)
            elif isinstance(val, datetime):
                ws.write_datetime(row_idx, col_idx, val, fmt)
            elif isinstance(val, int):
                ws.write_number(row_idx, col_idx, val, fmt)
            else:
                ws.write(row_idx, col_idx, str(val), fmt)

    ws.freeze_panes(1, 0)
    ws.autofilter(0, 0, len(records), len(ALL_COLS) - 1)
    print(f"   Sheet '{sheet_name}': {len(records):,} rows")


write_sheet(workbook, rta_deduped,      "RTA Email Body",  "#70AD47")
write_sheet(workbook, idadmins_deduped, "IDAdmins Reply",  "#2E75B6")
write_sheet(workbook, alert_deduped,    "Operation Alert", "#FF9900")

workbook.close()
print(f"\nExcel saved -> {OUTPUT_FILE}")
print(f"MSG saved  -> {MSG_BASE}")
print(f"   RTA_SEND/        : {len([r for r in rta_deduped      if r.get('SavedFile')]):,} files")
print(f"   IDADMINS_REPLY/  : {len([r for r in idadmins_deduped if r.get('SavedFile')]):,} files")
print(f"   OPERATION_ALERT/ : {len([r for r in alert_deduped    if r.get('SavedFile')]):,} files")

Connected  : huuchinh.nguyen@concentrix.com
Date range : 2026-01-01 -> 2026-05-25
Save MSG   : True -> C:\Users\ADMIN\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\DEACTIVATE_ACCOUNT
------------------------------------------------------------

[Sheet 1] RTA emails in 'RTA team'...
   Found: 2,019 items
   After filter: 26 items
   ++ 2026-02-22 | Hoang Minh Duc Nguyen | VN Expedia_ Account Deactivation Request_22-Feb-26
   ++ 2026-02-01 | Hoang Minh Duc Nguyen | VN Expedia_ Account Deactivation Request_01-Feb-26
   ++ 2026-01-06 | Duong Hoang Vu Pham | Expedia VN_ Account Deactivation Request_6-Jan-26
   ++ 2026-03-01 | Duong Hoang Vu Pham | Expedia VN_ Account Deactivation Request_1-Mar-26
   ++ 2026-03-06 | Hoang Minh Duc Nguyen | VN Expedia_ Account Deactivation Request_06-Mar-26
   ++ 2026-03-05 | Huu Chinh Nguyen | Expedia VN_ Account Deactivation Request_5-Mar-26
   ++ 2026-03-13 | Hoang Minh Duc Nguyen | VN Expedia_ Account Deactivation Request_13-Mar-26
   ++